In [1]:
import os
import random
import pandas as pd
from collections import defaultdict

# Đường dẫn đến thư mục chứa dữ liệu ảnh
data_path = '/kaggle/input/cs114-all-cars/'  # Thay đổi đường dẫn phù hợp
temp_path = '/kaggle/working/' #Nơi lưu file csv
# Bản đồ từ tên hiệu xe sang CategoryID
category_map = {
    "Others": 0,
    "Honda": 1,
    "Hyundai": 2,
    "KIA": 3,
    "Mazda": 4,
    "Mitsubishi": 5,
    "Suzuki": 6,
    "Toyota": 7,
    "VinFast": 8
}

# Số splits mặc định
NumSplits = 5

# Tìm tất cả các ảnh trong cây thư mục và tạo DataFrame
image_data = []
for brand in os.listdir(data_path):
    brand_path = os.path.join(data_path, brand)

    if os.path.isdir(brand_path) and brand in category_map:
        for file in os.listdir(brand_path):
            if file.endswith('.jpg'):
                image_full_path = f"{brand}/{file}"
                category_id = category_map[brand]
                image_data.append([image_full_path, category_id])

# Lưu tất cả ảnh vào CarDataset.csv
df_all = pd.DataFrame(image_data, columns=["ImageFullPath", "CategoryID"])
car_dataset_path = os.path.join(temp_path, "CarDataset.csv")
df_all.to_csv(car_dataset_path, index=False)
print(f"Đã lưu danh sách ảnh vào {car_dataset_path}")

# Chia dữ liệu thành các splits
splits = defaultdict(lambda: {"Train": [], "Test": []})

# Nhóm ảnh theo CategoryID
grouped = df_all.groupby("CategoryID")
for category_id, group in grouped:
    images = group.values.tolist()
    random.shuffle(images)  # Shuffle ngẫu nhiên

    # Chia ngẫu nhiên thành NumSplits phần
    fold_size = len(images) // NumSplits
    folds = [images[i * fold_size:(i + 1) * fold_size] for i in range(NumSplits)]

    # Xử lý dư (nếu không chia đều được)
    remainder = len(images) % NumSplits
    for i in range(remainder):
        folds[i].append(images[-(i + 1)])

    # Gom các folds thành Train/Test cho mỗi Split
    for i in range(NumSplits):
        test_set = folds[i]
        train_set = [img for fold in folds if fold != test_set for img in fold]

        splits[i]["Train"].extend(train_set)
        splits[i]["Test"].extend(test_set)

# Lưu các splits ra tệp
for i in range(NumSplits):
    train_path = os.path.join(temp_path, f"CarDataset-Splits-{i + 1}-Train.csv")
    test_path = os.path.join(temp_path, f"CarDataset-Splits-{i + 1}-Test.csv")

    train_df = pd.DataFrame(splits[i]["Train"], columns=["ImageFullPath", "CategoryID"])
    test_df = pd.DataFrame(splits[i]["Test"], columns=["ImageFullPath", "CategoryID"])

    train_df.to_csv(train_path, index=False)
    test_df.to_csv(test_path, index=False)

    print(f"Đã lưu Split-{i + 1} Train/Test vào {train_path} và {test_path}")

Đã lưu danh sách ảnh vào /kaggle/working/CarDataset.csv
Đã lưu Split-1 Train/Test vào /kaggle/working/CarDataset-Splits-1-Train.csv và /kaggle/working/CarDataset-Splits-1-Test.csv
Đã lưu Split-2 Train/Test vào /kaggle/working/CarDataset-Splits-2-Train.csv và /kaggle/working/CarDataset-Splits-2-Test.csv
Đã lưu Split-3 Train/Test vào /kaggle/working/CarDataset-Splits-3-Train.csv và /kaggle/working/CarDataset-Splits-3-Test.csv
Đã lưu Split-4 Train/Test vào /kaggle/working/CarDataset-Splits-4-Train.csv và /kaggle/working/CarDataset-Splits-4-Test.csv
Đã lưu Split-5 Train/Test vào /kaggle/working/CarDataset-Splits-5-Train.csv và /kaggle/working/CarDataset-Splits-5-Test.csv


In [2]:
print(df_all["CategoryID"].value_counts().sort_index())

# Thống kê và hiển thị
for i in range(NumSplits):
    train_df = pd.DataFrame(splits[i]["Train"], columns=["ImageFullPath", "CategoryID"])
    test_df = pd.DataFrame(splits[i]["Test"], columns=["ImageFullPath", "CategoryID"])

    print(f"\nSplit-{i + 1} Train - Thống kê CategoryID:")
    print(train_df["CategoryID"].value_counts().sort_index())

    print(f"\nSplit-{i + 1} Test - Thống kê CategoryID:")
    print(test_df["CategoryID"].value_counts().sort_index())

CategoryID
0    4564
1    3125
2    3424
3    3187
4    3169
5    2860
6    6552
7    5768
8    2813
Name: count, dtype: int64

Split-1 Train - Thống kê CategoryID:
CategoryID
0    3651
1    2500
2    2739
3    2549
4    2535
5    2288
6    5241
7    4614
8    2250
Name: count, dtype: int64

Split-1 Test - Thống kê CategoryID:
CategoryID
0     913
1     625
2     685
3     638
4     634
5     572
6    1311
7    1154
8     563
Name: count, dtype: int64

Split-2 Train - Thống kê CategoryID:
CategoryID
0    3651
1    2500
2    2739
3    2549
4    2535
5    2288
6    5241
7    4614
8    2250
Name: count, dtype: int64

Split-2 Test - Thống kê CategoryID:
CategoryID
0     913
1     625
2     685
3     638
4     634
5     572
6    1311
7    1154
8     563
Name: count, dtype: int64

Split-3 Train - Thống kê CategoryID:
CategoryID
0    3651
1    2500
2    2739
3    2550
4    2535
5    2288
6    5242
7    4614
8    2250
Name: count, dtype: int64

Split-3 Test - Thống kê CategoryID:
CategoryID
0 